# Predict — EuroBERT (from HuggingFace Hub)

Loads the fine-tuned EuroBERT model from HuggingFace Hub and generates predictions on any JSONL input file.
Output is a long-format parquet compatible with `orchestration.ipynb`.

**Input:** any `.jsonl` file with fields `akteId`, `text`, `rechtsfeitcodes`  
**Output:** `artifacts/predictions/{METHOD_NAME}.parquet`

In [ ]:
# --- settings ---
MODEL_REPO  = "ay44n04712/kadaster-eurobert"  # HuggingFace Hub repo with model + thresholds
METHOD_NAME = "eurobert"                       # name used in the output parquet
BATCH_SIZE  = 8                                # reduce if you run out of memory

# Input file — defaults to the held-out test set; swap for any JSONL with the same schema
import sys
sys.path.insert(0, "..")
from shared.config import TEST_PATH
INPUT_PATH = TEST_PATH

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN")

from shared.config  import DEVICE, ARTIFACTS, ROOT
from shared.config  import load_bert_config
from shared.data    import load_jsonl
from shared.labels  import get_mlb
from shared.io      import write_predictions

print(f"Device: {DEVICE}")

In [ ]:
# Load label space and input data
mlb, classes, num_classes = get_mlb()
df = load_jsonl(INPUT_PATH)
print(f"Input: {len(df):,} documents — {INPUT_PATH.name}")

In [ ]:
# Model architecture — must match what was used during training
class FineTunedMultiLabelClassifier(nn.Module):
    def __init__(self, num_classes: int, cfg):
        super().__init__()
        self.cfg        = cfg
        self.robbert    = AutoModel.from_pretrained(cfg.model_name, attn_implementation="sdpa")
        self.classifier = nn.Linear(self.robbert.config.hidden_size, num_classes)
        self.dropout    = nn.Dropout(0.1)

    def _pool(self, last_hidden_state, attention_mask, n_real_chunks):
        B = last_hidden_state.shape[0]
        doc_embeddings = []
        for i in range(B):
            real = last_hidden_state[i, :n_real_chunks[i]]
            mask = attention_mask[i, :n_real_chunks[i]]
            if self.cfg.pooling == "cls":
                doc_embeddings.append(real[:, 0, :].max(dim=0).values)
            elif self.cfg.pooling == "mean":
                mask_exp = mask.unsqueeze(-1).float()
                summed   = (real * mask_exp).sum(dim=(0, 1))
                count    = mask_exp.sum(dim=(0, 1)).clamp(min=1)
                doc_embeddings.append(summed / count)
            elif self.cfg.pooling == "max":
                mask_exp = mask.unsqueeze(-1).bool()
                real_masked = real.masked_fill(~mask_exp, float("-inf"))
                doc_embeddings.append(real_masked.flatten(0, 1).max(dim=0).values)
        return torch.stack(doc_embeddings)

    def forward(self, input_ids, attention_mask, n_chunks):
        B, max_chunks, seq_len = input_ids.shape
        ids_flat  = input_ids.view(B * max_chunks, seq_len)
        mask_flat = attention_mask.view(B * max_chunks, seq_len)
        outputs   = self.robbert(input_ids=ids_flat, attention_mask=mask_flat)
        hs_flat   = self.dropout(outputs.last_hidden_state)
        hs_all    = hs_flat.view(B, max_chunks, seq_len, -1)
        doc_emb   = self._pool(hs_all, attention_mask, n_chunks)
        return self.classifier(doc_emb)

print("Model class defined.")

In [ ]:
# Load config, build model architecture, overwrite with fine-tuned weights from HF Hub
cfg = load_bert_config(ROOT / "sweeps" / "eurobert.json")

print("Building model architecture...")
model = FineTunedMultiLabelClassifier(num_classes=num_classes, cfg=cfg)

print(f"Downloading fine-tuned weights from {MODEL_REPO}...")
weights_path = hf_hub_download(MODEL_REPO, "model.safetensors", token=HF_TOKEN)
model.load_state_dict(load_file(weights_path))
model.eval().to(DEVICE)
print("Model ready.")

In [ ]:
# Load per-class thresholds from HF Hub
thresh_path = hf_hub_download(MODEL_REPO, "thresholds.json", token=HF_TOKEN)
with open(thresh_path) as f:
    thresh_dict = json.load(f)

thresholds = np.array([thresh_dict[str(c)] for c in classes])
print(f"Thresholds loaded — min={thresholds.min():.2f}  max={thresholds.max():.2f}  mean={thresholds.mean():.2f}")

In [ ]:
# Tokenize and chunk documents
def pretokenize(texts: list, cfg, tokenizer) -> list:
    CLS_ID = tokenizer.cls_token_id
    SEP_ID = tokenizer.sep_token_id
    PAD_ID = tokenizer.pad_token_id
    all_chunks = []
    for text in tqdm(texts, desc="Pre-tokenizing"):
        input_ids = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"][0].tolist()
        tokens = input_ids[1:-1]
        stride = cfg.stride if cfg.stride > 0 else cfg.content_size
        starts = list(range(0, max(len(tokens), 1), stride))
        chunks = [tokens[s : s + cfg.content_size] for s in starts][:cfg.max_chunks]
        all_ids, all_masks = [], []
        for chunk in chunks:
            ids     = [CLS_ID] + chunk + [SEP_ID]
            mask    = [1] * len(ids)
            pad_len = cfg.content_size + 2 - len(ids)
            ids    += [PAD_ID] * pad_len
            mask   += [0]      * pad_len
            all_ids.append(ids)
            all_masks.append(mask)
        all_chunks.append((
            torch.tensor(all_ids,   dtype=torch.long),
            torch.tensor(all_masks, dtype=torch.long),
        ))
    return all_chunks


class InferenceDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, idx):
        input_ids, attention_mask = self.chunks[idx]
        return {"input_ids": input_ids, "attention_mask": attention_mask}


def collate_fn(batch):
    seq_len    = batch[0]["input_ids"].shape[1]
    max_chunks = max(item["input_ids"].shape[0] for item in batch)
    PAD_ID     = 1
    ids_batch, mask_batch, n_chunks_batch = [], [], []
    for item in batch:
        n   = item["input_ids"].shape[0]
        pad = max_chunks - n
        ids_batch.append(torch.cat([item["input_ids"],       torch.full((pad, seq_len), PAD_ID, dtype=torch.long)], dim=0))
        mask_batch.append(torch.cat([item["attention_mask"], torch.zeros((pad, seq_len), dtype=torch.long)],        dim=0))
        n_chunks_batch.append(n)
    return {
        "input_ids"      : torch.stack(ids_batch),
        "attention_mask" : torch.stack(mask_batch),
        "n_chunks"       : torch.tensor(n_chunks_batch),
    }


tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, token=HF_TOKEN)
chunks    = pretokenize(df["text"].tolist(), cfg, tokenizer)
dataset   = InferenceDataset(chunks)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
print(f"Tokenized {len(dataset):,} documents into {sum(len(c[0]) for c in chunks):,} total chunks.")

In [ ]:
# Run inference
all_probs = []
with torch.inference_mode():
    for batch in tqdm(loader, desc="Inference"):
        logits = model(
            input_ids      = batch["input_ids"].to(DEVICE),
            attention_mask = batch["attention_mask"].to(DEVICE),
            n_chunks       = batch["n_chunks"].to(DEVICE),
        )
        all_probs.append(torch.sigmoid(logits).cpu().float().numpy())

probs = np.vstack(all_probs)   # (N, num_classes)
print(f"Inference complete — output shape: {probs.shape}")

In [ ]:
# Apply per-class thresholds and write parquet
predicted = (probs > thresholds[np.newaxis, :]).astype(bool)

out_path = write_predictions(
    akteIds   = df["akteId"].tolist(),
    scores    = probs,
    predicted = predicted,
    classes   = classes,
    method    = METHOD_NAME,
)

preds_per_doc = predicted.sum(axis=1)
print(f"Deeds with ≥1 predicted code : {(preds_per_doc > 0).sum():,} / {len(df):,}")
print(f"Mean codes per deed          : {preds_per_doc.mean():.2f}")
print(f"\nReady for orchestration.ipynb → {out_path}")